# 00 — Serviços AWS e prontidão

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 1 conta AWS

Esta aula não exige fronteiras entre contas. Seus stacks, integrações e permissões precisam estar implantados na conta da plataforma. Ela não funciona automaticamente em uma conta AWS vazia.

**Runner CodeBuild do projeto:** a seleção é declarada no deploy. A jornada `single-account` inclui 00–09 e 15–16; Registry, Terraform e RAG são exigidos somente pelas aulas que os utilizam. Ampliar a seleção exige atualizar a implantação pelo IaC. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 00](../docs/diagramas/svg/00_preflight.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "00")
print({"lab": "00", "dependencies": list(dependencies), "region": config["region"]})

### Conectar ao ambiente implantado

Lemos os outputs exatos do CloudFormation. O cliente não cria infraestrutura e não substitui um serviço indisponível.

In [ ]:
lab = AWSWorkshop(CONFIG, domain="platform")
print(lab.preflight())

### Ler o Gateway pela API de controle

Gateway reúne protocolo MCP, autenticação e Policy. READY indica prontidão; ENFORCE torna a decisão Cedar obrigatória antes de invocar um target.

In [ ]:
control = lab.aws.client("bedrock-agentcore-control")
gateway = control.get_gateway(gatewayIdentifier=lab.gateway["GatewayId"])
print({key: gateway[key] for key in ["gatewayArn", "status", "authorizerType",
                                    "policyEngineConfiguration"]})
assert gateway["status"] == "READY"
assert gateway["policyEngineConfiguration"]["mode"] == "ENFORCE"

### Verificar a fronteira de autenticação

Enviamos um initialize MCP sem Authorization. Só 401/403 comprovam a negativa; timeout ou erro de servidor fazem o teste falhar.

In [ ]:
display(Code(inspect.getsource(lab.deny_anonymous), language="python"))
print(await lab.deny_anonymous())

### Rejeitar um JWT inválido

Um texto com formato de token não estabelece uma identidade. Exigimos a negativa HTTP do authorizer e o ID nativo da requisição.

In [ ]:
import httpx
async with httpx.AsyncClient(timeout=30, follow_redirects=False) as http:
    response = await http.post(lab.gateway["GatewayUrl"],
        headers={"Authorization": "Bearer invalid.token.value"},
        json={"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}})
assert response.status_code in {401, 403}
assert response.headers.get("x-amzn-requestid")
print({"invalid_jwt": "DENIED", "http_status": response.status_code,
       "gateway_request_id": response.headers["x-amzn-requestid"]})

### Comprovar que o Runtime exige a role do Gateway

O preflight confere a resource policy e a trust policy pelo IAM. Agora tentamos uma invocação direta com a role do runner: o serviço deve negar antes de executar o servidor MCP.

In [ ]:
display(Code(inspect.getsource(lab.deny_direct_runtime), language="python"))
print(lab.deny_direct_runtime())

**Observe:** IAM controla quem administra os recursos; o authorizer JWT
autentica chamadas MCP; Cedar autoriza pessoa, aplicação, ação e entrada.
O Gateway não transforma uma permissão de descoberta em permissão de execução.

**Entra ID:** o mesmo desenho usa discovery URL, chaves públicas e audiência
do Entra. As claims `tid`, `oid`, `azp` e os consentimentos precisam de
regras próprias. Não copie os valores Cognito para essas claims.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.